Caveat: Dataset does not establish wether customer cancellations occured after a driver accepted!

Prediction Target: binary customer cancellation model (completed = 0; cancelled by customer = 1)
this is because we are trying to help drivers avoid spending time travelling toward a customer who subsequently cancels.

In [7]:
import pandas as pd

df = pd.read_csv("../data/uberData.csv")

print("Original shape:", df.shape)
display(df.head())

Original shape: (150000, 21)


,Date,Time,Booking ID,Booking Status,Customer ID,Vehicle Type,Pickup Location,Drop Location,Avg VTAT,Avg CTAT,...,Reason for cancelling by Customer,Cancelled Rides by Driver,Driver Cancellation Reason,Incomplete Rides,Incomplete Rides Reason,Booking Value,Ride Distance,Driver Ratings,Customer Rating,Payment Method
0,2024-03-23,12:29:38,"""CNR5884300""",No Driver Found,"""CID1982111""",eBike,Palam Vihar,Jhilmil,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2024-11-29,18:01:39,"""CNR1326809""",Incomplete,"""CID4604802""",Go Sedan,Shastri Nagar,Gurgaon Sector 56,4.9,14.0,...,NaN,NaN,NaN,1.0,Vehicle Breakdown,237.0,5.73,NaN,NaN,UPI
2,2024-08-23,08:56:10,"""CNR8494506""",Completed,"""CID9202816""",Auto,Khandsa,Malviya Nagar,13.4,25.8,...,NaN,NaN,NaN,NaN,NaN,627.0,13.58,4.9,4.9,Debit Card
3,2024-10-21,17:17:25,"""CNR8906825""",Completed,"""CID2610914""",Premier Sedan,Central Secretariat,Inderlok,13.1,28.5,...,NaN,NaN,NaN,NaN,NaN,416.0,34.02,4.6,5.0,UPI
4,2024-09-16,22:08:00,"""CNR1950162""",Completed,"""CID9933542""",Bike,Ghitorni Village,Khan Market,5.3,19.6,...,NaN,NaN,NaN,NaN,NaN,737.0,48.21,4.1,4.3,UPI


In [8]:
target_map = {
    "Completed": 0,
    "Cancelled by Customer": 1
}

df["target"] = df["Booking Status"].map(target_map)

# Keep only the two outcomes we want to predict
df = df.dropna(subset=["target"]).copy()

df["target"] = df["target"].astype(int)

print("Remaining rows:", len(df))
print(df["target"].value_counts())
print(df["target"].value_counts(normalize=True).round(3))

Remaining rows: 103500
target
0    93000
1    10500
Name: count, dtype: int64
target
0    0.899
1    0.101
Name: proportion, dtype: float64


In [9]:
# Convert date and time into useful features
df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
df["Hour"] = pd.to_datetime(
    df["Time"], format="%H:%M:%S", errors="coerce"
).dt.hour

df["DayOfWeek"] = df["Date"].dt.dayofweek
df["Month"] = df["Date"].dt.month

features = [
    "Vehicle Type",
    "Pickup Location",
    "Drop Location",
    "Hour",
    "DayOfWeek",
    "Month"
]

X = df[features].copy()
y = df["target"]

display(X.head())

,Vehicle Type,Pickup Location,Drop Location,Hour,DayOfWeek,Month
2,Auto,Khandsa,Malviya Nagar,8,4,8
3,Premier Sedan,Central Secretariat,Inderlok,17,0,10
4,Bike,Ghitorni Village,Khan Market,22,0,9
5,Auto,AIIMS,Narsinghpur,9,1,2
6,Go Mini,Vaishali,Punjabi Bagh,15,0,6


In [ ]:
from sklearn.model_selection import train_test_split

# might also do chronological split to provide more realistic evaluation later. 
# We will use current split for baseline model for now
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print("Training:", X_train.shape)
print("Testing:", X_test.shape)

Training: (82800, 6)
Testing: (20700, 6)


In [11]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

numeric_features = [
    "Hour",
    "DayOfWeek",
    "Month"
]

categorical_features = [
    "Vehicle Type",
    "Pickup Location",
    "Drop Location"
]

numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(
        strategy="most_frequent"
    )),
    ("encoder", OneHotEncoder(
        handle_unknown="ignore"
    ))
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_transformer, numeric_features),
    ("categorical", categorical_transformer, categorical_features)
])

In [12]:
X_train_processed = preprocessor.fit_transform(X_train)
X_test_processed = preprocessor.transform(X_test)

print("Training shape:", X_train_processed.shape)
print("Testing shape:", X_test_processed.shape)

Training shape: (82800, 362)
Testing shape: (20700, 362)
